# 07 MedSAM mask refinement (Phase 0) — keep V6 boxes, swap the mask

**Evaluation only. Zero training.** Phase 0 of `docs/medsam_refine_research_notes.md`.

## The idea (and why it is NOT Phase 1c again)

V6 localizes the **large** classes (Abrasion/Crown/Filling) well — that is where the mAP weight is —
but YOLO-seg's prototype masks are coarse (`imgsz/4`). `mAP50-95` is IoU-strict, so coarse masks
leave AP on the table in the 0.7–0.95 band. This notebook keeps V6's **box + class + confidence**
unchanged and only **replaces the mask** with a MedSAM mask (box-prompted). Unlike Phase 1c, it
refines masks where the boxes are *trustworthy* and needs **no training**.

## What it computes (all with one in-notebook evaluator → clean apples-to-apples)

For the **same** V6 boxes / classes / confidences, scored by the identical mask-mAP matcher:

- `v6_native@conf` — V6 boxes + **V6's own masks** (the in-notebook baseline).
- `v6box_medsam@conf` — V6 boxes + **MedSAM masks** (the real pipeline). *The headline delta is this
  vs `v6_native` — the mask is the ONLY variable.*
- `TPonly_*@0.05` — same, but only V6 boxes matching a GT (perfect FP rejection) → isolates mask
  quality on true positives.
- `oracle_medsam` — **GT boxes** + MedSAM masks + GT class → the ceiling of "perfect localization +
  MedSAM mask" (decouples mask quality from V6 box quality, the same oracle discipline as src/04).

## Pre-registered reading rules (from the research note)

- Decision metric = the comparable full-image mask-mAP here; **the headline is the LARGE classes**
  (Abrasion/Crown/Filling) — that is where the hypothesis and the mAP weight live.
- Small Caries are reported but low-weight/noisy (Caries 4 n=4 / Caries 6 n=5 are noise; only
  Caries 1/2/5 are usable as a trend). Treat mAP deltas under ~0.003 as noise.
- **Go/no-go:** `v6box_medsam` (real boxes) must beat `v6_native` beyond the 0.003 band on the large
  classes (and not regress the aggregate) to be worth a submission. `oracle_medsam` is an upper bound.


## 1. Setup

Ultralytics (V6 = Stage 1) + `segment_anything` (MedSAM loads into the `vit_b` registry).
Training/eval kernels usually have Internet on; if not, add a `segment-anything` wheel + the MedSAM
checkpoint as offline Kaggle inputs.

In [ ]:
try:
    import ultralytics  # noqa
    import segment_anything  # noqa
    print("ultralytics + segment_anything already present")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "ultralytics",
                           "git+https://github.com/facebookresearch/segment-anything.git"])
    print("installed ultralytics + segment-anything")

In [ ]:
import os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import yaml
import torch
import torch.nn.functional as F
from ultralytics import YOLO
from segment_anything import sam_model_registry

cv2.setNumThreads(0)            # avoid cv2-thread contention in tight loops
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 2. Configuration

`USE_ROI_CROP=True` crops the (padded) box ROI from the **original-resolution** image and runs MedSAM
on that crop — SAM internally resizes to 1024², so a tiny lesion in a full panoramic has almost no
effective resolution; cropping restores it (the context-vs-effective-resolution knob, same family as
the src/04 padding ablation). Set `False` to prompt MedSAM on the full image (faster: one image
encode per image instead of per box, fine for large objects).

In [ ]:
# ---- MedSAM ----
MODEL_TYPE     = "vit_b"        # MedSAM is a fine-tuned SAM ViT-B
USE_ROI_CROP   = True           # True: crop padded ROI per box (better for small lesions); False: full image
PAD_MODE       = "relative"     # ROI padding: "relative" (x box size) or "absolute" (px)
PAD_FACTOR     = 1.5            # used when PAD_MODE == "relative"
PAD_ABS_PX     = 48             # used when PAD_MODE == "absolute"
SAM_INPUT      = 1024           # SAM image-encoder input side (fixed by the architecture)

# ---- Stage 1 (V6) detector ----
IMG_SIZE_DET   = 768            # V6 was trained at 768
CAPTURE_CONF   = 0.05           # capture V6 boxes once at this conf, then filter in python
DET_IOU        = 0.60           # YOLO NMS IoU
MAX_DET        = 300

# ---- Eval ----
PIPE_CONFS     = [0.05, 0.25]                  # conf thresholds to score the pipeline at
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)    # mask-AP thresholds (same as src/03/04/05)

# V6 per-class Mask mAP50-95 reference from the src/03 same-code run (CONTEXT ONLY — the real
# baseline is `v6_native` re-scored in THIS notebook with the identical matcher).
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

# Class names treated as "large" for the headline summary (case-insensitive substring match).
LARGE_CLASSES = ["abrasion", "crown", "filling"]

print("MedSAM", MODEL_TYPE, "| ROI crop:", USE_ROI_CROP, "| pad", PAD_MODE, PAD_FACTOR,
      "| det imgsz", IMG_SIZE_DET, "| capture conf", CAPTURE_CONF)

## 3. Locate the dataset + validation ground truth

Same auto-detection as src/04/05: find `yolo_seg_train.yaml`, read class names, resolve the val
split, parse the polygon labels.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

val_images = resolve_split(dcfg.get("val", dcfg.get("valid")))

def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try:
        cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

val_objs = {}   # image_path -> list of (cls, poly_norm)
lbl_dir = labels_dir_for(val_images)
for ip in sorted(p for p in Path(val_images).iterdir() if p.suffix.lower() in IMG_EXTS):
    lp = lbl_dir / (ip.stem + ".txt")
    objs = []
    if lp.exists():
        for line in lp.read_text().splitlines():
            pr = parse_seg_line(line)
            if pr is not None: objs.append(pr)
    val_objs[str(ip)] = objs

n_gt = np.zeros(num_classes, dtype=int)
for objs in val_objs.values():
    for c, _ in objs: n_gt[c] += 1
print("classes:", names)
print("val images:", len(val_objs), "| total GT objects:", int(n_gt.sum()))
for c in range(num_classes):
    print(f"  {names[c]:>14s}: n_gt={int(n_gt[c])}")

## 4. Load Stage 1 (V6 detector) + MedSAM

V6 is auto-detected by the `version6` filename keyword (the MedSAM file is excluded so they never
collide). MedSAM is loaded into the `vit_b` registry and run in **inference only** (no training).

In [ ]:
# Optional hard overrides — set to exact /kaggle/input paths if auto-detection picks wrong.
MANUAL_V6_PATH     = None
MANUAL_MEDSAM_PATH = None

def find_pt(include, exclude=(), exts=(".pt", ".pth")):
    cands = []
    for p in Path("/kaggle/input").rglob("*"):
        if p.suffix.lower() not in exts: continue
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if "best" in p.name.lower() else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6_cands     = find_pt(["version6", "v6", "yolov8s", "img768"], exclude=["medsam", "sam_vit", "stage2"])
medsam_cands = find_pt(["medsam", "sam_vit_b", "sam-vit-b", "sam_b"], exclude=["version", "yolo", "stage2"])
print("V6 candidates:");     [print("  ", p) for p in v6_cands[:5]]
print("MedSAM candidates:"); [print("  ", p) for p in medsam_cands[:5]]

V6_PATH     = Path(MANUAL_V6_PATH) if MANUAL_V6_PATH else (v6_cands[0] if v6_cands else None)
MEDSAM_PATH = Path(MANUAL_MEDSAM_PATH) if MANUAL_MEDSAM_PATH else (medsam_cands[0] if medsam_cands else None)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector .pt found. Add it as a Kaggle input or set MANUAL_V6_PATH.")
if MEDSAM_PATH is None or not MEDSAM_PATH.exists():
    raise FileNotFoundError("No MedSAM checkpoint found. Add medsam_vit_b.pth as a Kaggle input or set MANUAL_MEDSAM_PATH.")
print("\nUsing V6    :", V6_PATH)
print("Using MedSAM:", MEDSAM_PATH)

detector = YOLO(str(V6_PATH))

# Load MedSAM weights into the SAM vit_b architecture (robust to a few checkpoint layouts).
sam = sam_model_registry[MODEL_TYPE]()
sd = torch.load(str(MEDSAM_PATH), map_location="cpu")
if isinstance(sd, dict):
    sd = sd.get("model", sd.get("state_dict", sd))
missing, unexpected = sam.load_state_dict(sd, strict=False)
sam = sam.to(DEVICE).eval()
print("\nMedSAM loaded. missing:", len(missing), "unexpected:", len(unexpected))
if len(missing) > 50:
    print("  WARNING: many missing keys — check this is a ViT-B MedSAM/SAM checkpoint.")

## 5. Run V6 on the validation images (boxes + conf + class + native mask)

One detector pass per image at `CAPTURE_CONF`, capturing each detection's box, confidence, class and
its **native YOLO mask polygon** (so `v6_native` can be scored with the same matcher as the MedSAM
variants). Boxes are reused for the conf sweep (no re-inference).

In [ ]:
det_by_image = {}   # path -> list of dict(box, conf, cls, poly_px or None)
wh_by_image = {}
for ip in val_objs:
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]; wh_by_image[ip] = (w, h)
    res = detector.predict(source=img, imgsz=IMG_SIZE_DET, conf=CAPTURE_CONF, iou=DET_IOU,
                           max_det=MAX_DET, device=DEVICE, task="segment",
                           verbose=False, save=False)[0]
    dets = []
    if res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy()
        conf = res.boxes.conf.cpu().numpy()
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        polys = res.masks.xy if (res.masks is not None) else [None] * len(xyxy)
        for b, cf, cl, pg in zip(xyxy, conf, cls, polys):
            x0, y0, x1, y1 = [int(round(v)) for v in b]
            x0 = max(0, min(x0, w-1)); y0 = max(0, min(y0, h-1))
            x1 = max(x0+1, min(x1, w)); y1 = max(y0+1, min(y1, h))
            poly_px = np.asarray(pg, dtype=np.float64) if (pg is not None and len(pg) >= 3) else None
            dets.append({"box": (x0, y0, x1, y1), "conf": float(cf), "cls": int(cl), "poly_px": poly_px})
    det_by_image[ip] = dets
tot = sum(len(d) for d in det_by_image.values())
print(f"captured {tot} V6 boxes over {len(det_by_image)} images at conf>={CAPTURE_CONF}")

## 6. Geometry + MedSAM inference helpers

`medsam_masks_for_boxes` runs MedSAM (box-prompted) for every box of one image and returns, per box,
a `(global_bbox, local_mask)` pair — the bbox of the predicted mask plus the mask rasterized inside
it (the same local-mask representation the src/04 evaluator consumes). Two paths:

- **ROI-crop mode** (`USE_ROI_CROP=True`): crop the padded square ROI from the original image, encode
  *that* crop at 1024², decode the (mapped) box prompt, place the mask back by the crop offset.
- **Full-image mode**: encode the whole image once, decode all boxes against that single embedding.

MedSAM preprocessing = min-max normalise the 1024² image to [0,1] (its training convention), fed
straight to `image_encoder` (we deliberately bypass SAM's pixel mean/std).

In [ ]:
def padded_square_box(x0, y0, x1, y1, w, h):
    bw, bh = x1 - x0, y1 - y0
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    half = (max(bw, bh) * (1.0 + PAD_FACTOR) / 2.0) if PAD_MODE == "relative" else (max(bw, bh) / 2.0 + PAD_ABS_PX)
    half = max(half, 1.0)
    sx0, sy0, sx1, sy1 = cx - half, cy - half, cx + half, cy + half
    sx0 = max(0.0, min(sx0, w - 1)); sy0 = max(0.0, min(sy0, h - 1))
    sx1 = max(sx0 + 1, min(sx1, w)); sy1 = max(sy0 + 1, min(sy1, h))
    return int(round(sx0)), int(round(sy0)), int(round(sx1)), int(round(sy1))

@torch.no_grad()
def medsam_encode(img_rgb):
    # min-max normalise the 1024 image to [0,1] (MedSAM convention) and run the image encoder.
    img = cv2.resize(img_rgb, (SAM_INPUT, SAM_INPUT), interpolation=cv2.INTER_CUBIC).astype(np.float32)
    mn, mx = float(img.min()), float(img.max())
    img = (img - mn) / max(mx - mn, 1e-8)
    t = torch.as_tensor(img).permute(2, 0, 1)[None].to(DEVICE)
    return sam.image_encoder(t)                       # (1,256,64,64)

@torch.no_grad()
def medsam_decode(emb, boxes_xyxy, src_hw):
    # boxes in the SOURCE-image (or crop) pixel frame; scale to 1024 and decode (multimask off).
    H, W = src_hw
    scale = np.array([SAM_INPUT / W, SAM_INPUT / H, SAM_INPUT / W, SAM_INPUT / H], dtype=np.float32)
    box_t = torch.as_tensor(np.asarray(boxes_xyxy, dtype=np.float32) * scale, device=DEVICE)
    sparse, dense = sam.prompt_encoder(points=None, boxes=box_t, masks=None)
    low_res, _ = sam.mask_decoder(
        image_embeddings=emb,                         # (1,256,64,64) broadcasts over the box batch
        image_pe=sam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
    )
    masks = F.interpolate(low_res, size=(H, W), mode="bilinear", align_corners=False)
    return (torch.sigmoid(masks)[:, 0] > 0.5).cpu().numpy().astype(np.uint8)   # (n,H,W)

def mask_to_local(mask, off_x=0, off_y=0):
    # Tight bbox of the binary mask (+offset) and the mask cropped to it; None if empty.
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None
    x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    local = mask[y0:y1, x0:x1].astype(np.uint8)
    return (x0 + off_x, y0 + off_y, x1 + off_x, y1 + off_y), local

def medsam_masks_for_boxes(img_rgb, boxes):
    # Returns list aligned with `boxes`: (global_bbox, local_mask) or None per box.
    h, w = img_rgb.shape[:2]
    out = [None] * len(boxes)
    if not boxes:
        return out
    if USE_ROI_CROP:
        for i, (x0, y0, x1, y1) in enumerate(boxes):
            sx0, sy0, sx1, sy1 = padded_square_box(x0, y0, x1, y1, w, h)
            crop = img_rgb[sy0:sy1, sx0:sx1]
            ch, cw = crop.shape[:2]
            emb = medsam_encode(crop)
            box_in_crop = [[x0 - sx0, y0 - sy0, x1 - sx0, y1 - sy0]]
            m = medsam_decode(emb, box_in_crop, (ch, cw))[0]   # crop-sized mask
            out[i] = mask_to_local(m, off_x=sx0, off_y=sy0)
    else:
        emb = medsam_encode(img_rgb)
        masks = medsam_decode(emb, [list(b) for b in boxes], (h, w))  # (n,H,W) full-image masks
        for i in range(len(boxes)):
            out[i] = mask_to_local(masks[i])
    return out

def poly_px_to_local(poly_px, w, h):
    # Rasterize a YOLO mask polygon (pixel coords) into its own tight bbox frame.
    if poly_px is None or len(poly_px) < 3:
        return None
    pts = poly_px.copy()
    gx0 = max(0, int(np.floor(pts[:, 0].min()))); gy0 = max(0, int(np.floor(pts[:, 1].min())))
    gx1 = min(w, int(np.ceil(pts[:, 0].max())));  gy1 = min(h, int(np.ceil(pts[:, 1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

print("helpers ready | ROI crop:", USE_ROI_CROP)

## 7. Cache masks once (V6 native + MedSAM-on-V6-boxes + MedSAM-oracle)

The expensive MedSAM passes run once here; every scored variant in §9 is then a cheap filter over
these caches. `matches_gt` (box-IoU ≥ 0.5 with any GT) is recorded for the TP-only variant.

In [ ]:
def gt_pixel_bbox(poly_norm, w, h):
    xs = poly_norm[:, 0] * w; ys = poly_norm[:, 1] * h
    return (max(0, int(np.floor(xs.min()))), max(0, int(np.floor(ys.min()))),
            min(w, int(np.ceil(xs.max()))), min(h, int(np.ceil(ys.max()))))

def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1]); bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

# --- V6 cache: native mask + MedSAM mask for every captured V6 box ---
v6_cache = {}     # ip -> list of dict(conf, cls, matches_gt, native, medsam)  (native/medsam = (gbox,lmask) or None)
n_imgs = len(val_objs)
for k, (ip, objs) in enumerate(val_objs.items(), 1):
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    w, h = wh_by_image[ip]
    dets = det_by_image[ip]
    gt_boxes = [gt_pixel_bbox(p, w, h) for _, p in objs]
    medsam = medsam_masks_for_boxes(img_rgb, [d["box"] for d in dets])
    items = []
    for d, ms in zip(dets, medsam):
        items.append({
            "conf": d["conf"], "cls": d["cls"],
            "matches_gt": any(box_iou(d["box"], gb) >= 0.5 for gb in gt_boxes),
            "native": poly_px_to_local(d["poly_px"], w, h),
            "medsam": ms,
        })
    v6_cache[ip] = items
    if k % 20 == 0 or k == n_imgs:
        print(f"  V6-box MedSAM cache: {k}/{n_imgs} images", flush=True)

# --- Oracle cache: MedSAM mask for every GT box (class = GT class, conf = 1.0) ---
oracle_cache = {}  # ip -> list of dict(cls, medsam)
for k, (ip, objs) in enumerate(val_objs.items(), 1):
    if not objs:
        oracle_cache[ip] = []; continue
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    w, h = wh_by_image[ip]
    gt_boxes = [gt_pixel_bbox(p, w, h) for _, p in objs]
    medsam = medsam_masks_for_boxes(img_rgb, gt_boxes)
    oracle_cache[ip] = [{"cls": objs[i][0], "medsam": medsam[i]} for i in range(len(objs))]
    if k % 20 == 0 or k == n_imgs:
        print(f"  oracle MedSAM cache: {k}/{n_imgs} images", flush=True)

print("caches built.")

## 8. Mask-mAP evaluator (identical matcher to src/04)

`gt_local_mask` rasterizes each GT polygon in its own bbox frame; `iou_local` is the exact mask IoU
over the overlap rectangle; `ap_101` is the 101-point AP. `score_variant` takes a per-image list of
`(cls, score, gbox, lmask)` predictions and returns the per-class AP averaged over the 10 IoU
thresholds — the same numbers as src/03/04/05.

In [ ]:
def gt_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

# Pre-rasterize GT once per image (reused by every variant).
gt_local_by_image = {}
for ip, objs in val_objs.items():
    w, h = wh_by_image[ip]
    gt_local_by_image[ip] = [(c, *gt_local_mask(p, w, h)) for c, p in objs]

def score_variant(preds_by_image):
    # preds_by_image: ip -> list of (cls, score, gbox, lmask). Returns per-class AP (len num_classes).
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip, gts in gt_local_by_image.items():
        preds = preds_by_image.get(ip, [])
        # cache IoU once, then threshold (same as src/04).
        iou_mat = np.zeros((len(preds), len(gts)), dtype=np.float32)
        for pi, (pc, sc, pbox, pm) in enumerate(preds):
            for gi, (gc, gbox, gm) in enumerate(gts):
                if pc == gc:
                    iou_mat[pi, gi] = iou_local(pbox, pm, gbox, gm)
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(preds)), key=lambda i: preds[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc = preds[pi][0]
                best, bg = 0.0, -1
                for gi in range(len(gts)):
                    if used[gi] or gts[gi][0] != pc: continue
                    v = iou_mat[pi, gi]
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((preds[pi][1], tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                    [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

print("evaluator ready.")

## 9. Build the variants and score them

`v6_native` vs `v6box_medsam` share identical boxes/classes/confidences — so their per-class delta
is the **pure mask-quality effect**, the whole point of this experiment.

In [ ]:
def v6_preds(use_medsam, conf_t, tp_only=False):
    out = {}
    key = "medsam" if use_medsam else "native"
    for ip, items in v6_cache.items():
        preds = []
        for it in items:
            if it["conf"] < conf_t: continue
            if tp_only and not it["matches_gt"]: continue
            md_ = it[key]
            if md_ is None: continue
            (gbox, lmask) = md_
            preds.append((it["cls"], it["conf"], gbox, lmask))
        out[ip] = preds
    return out

def oracle_preds():
    out = {}
    for ip, items in oracle_cache.items():
        preds = []
        for it in items:
            if it["medsam"] is None: continue
            gbox, lmask = it["medsam"]
            preds.append((it["cls"], 1.0, gbox, lmask))   # GT class, constant conf (ceiling)
        out[ip] = preds
    return out

variants = {}
for conf_t in PIPE_CONFS:
    variants[f"v6_native@{conf_t}"]    = score_variant(v6_preds(False, conf_t))
    variants[f"v6box_medsam@{conf_t}"] = score_variant(v6_preds(True,  conf_t))
variants["TPonly_v6_native@0.05"]    = score_variant(v6_preds(False, 0.05, tp_only=True))
variants["TPonly_v6box_medsam@0.05"] = score_variant(v6_preds(True,  0.05, tp_only=True))
variants["oracle_medsam"]            = score_variant(oracle_preds())

def agg(pac):
    v = ~np.isnan(pac)
    return float(np.nanmean(pac[v])) if v.any() else float("nan")

def large_agg(pac):
    idx = [c for c in range(num_classes)
           if any(L in names[c].lower() for L in LARGE_CLASSES) and not np.isnan(pac[c])]
    return float(np.mean([pac[c] for c in idx])) if idx else float("nan")

print("=== aggregate Mask mAP50-95 (in-notebook matcher; V6 src/03 ref = %.4f) ===" % V6_REF_MAP)
for tag, pac in variants.items():
    print(f"  {tag:>26s}:  all={agg(pac):.4f}   large(Abr/Crown/Fill)={large_agg(pac):.4f}")

# Headline mask-only delta: same V6 boxes, MedSAM mask vs YOLO mask.
for conf_t in PIPE_CONFS:
    nat = variants[f"v6_native@{conf_t}"]; med = variants[f"v6box_medsam@{conf_t}"]
    print(f"\n--- per-class: v6box_medsam@{conf_t} vs v6_native@{conf_t} (mask is the only change) ---")
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        d = med[c] - nat[c] if not (np.isnan(med[c]) or np.isnan(nat[c])) else float("nan")
        tag = "  <-- LARGE" if any(L in names[c].lower() for L in LARGE_CLASSES) else ""
        print(f"  {names[c]:>14s} n={int(n_gt[c]):>3d}  native={nat[c]:.3f}  medsam={med[c]:.3f}  d={d:+.3f}{tag}")

## 10. How to read this → go / no-go

- **Headline = the large classes** (Abrasion/Crown/Filling) on `v6box_medsam@0.05` vs `v6_native@0.05`.
  Same boxes/classes/scores, so any difference is the mask. A clear **positive** delta beyond ~0.003
  means MedSAM masks are sharper at strict IoU where the mAP weight is → worth a submission path.
- **`oracle_medsam`** is the ceiling (perfect GT-box localization + MedSAM mask). If even the oracle
  does not beat `v6_native`, MedSAM's masks are not better on this domain (dental pano X-ray is
  partly OOD for MedSAM) → consider the optional decoder-only fine-tune in the research note, or stop.
- **Watch Crown** for the SAM failure mode (segmenting the whole tooth instead of the crown); if a
  large class regresses, route it back to the V6 mask (per-class, decided after seeing the numbers).
- Re-run §2 with `USE_ROI_CROP=False` (full-image prompt) and/or different `PAD_FACTOR` to check
  sensitivity. Small Caries are low-weight/noisy — don't let them drive the decision.

## 11. Save outputs (Kaggle "Save Version")

Writes a per-class table (every variant) + a self-describing summary to `/kaggle/working`, so a
background commit run leaves downloadable files. Archive these into the repo `stage2/` (or a new
`medsam/`) folder as `medsam_phase0_*` alongside the other results.

In [ ]:
rows = []
for tag, pac in variants.items():
    for c in range(num_classes):
        v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
        rows.append({"variant": tag, "class": names[c], "n_gt": int(n_gt[c]),
                     "AP": (None if np.isnan(pac[c]) else float(pac[c])),
                     "v6_ref_AP_src03": (None if np.isnan(v6) else float(v6))})
results_df = pd.DataFrame(rows)
RESULTS_CSV = "/kaggle/working/medsam_phase0_results.csv"
results_df.to_csv(RESULTS_CSV, index=False)

def _f(x):
    return None if (x is None or (isinstance(x, float) and np.isnan(x))) else float(x)

summary = {
    "config": {
        "MODEL_TYPE": MODEL_TYPE, "USE_ROI_CROP": USE_ROI_CROP, "PAD_MODE": PAD_MODE,
        "PAD_FACTOR": PAD_FACTOR, "PAD_ABS_PX": PAD_ABS_PX, "IMG_SIZE_DET": IMG_SIZE_DET,
        "CAPTURE_CONF": CAPTURE_CONF, "PIPE_CONFS": PIPE_CONFS, "SEED": SEED,
    },
    "aggregate_mAP50_95": {tag: _f(agg(pac)) for tag, pac in variants.items()},
    "large_class_mAP50_95": {tag: _f(large_agg(pac)) for tag, pac in variants.items()},
    "v6_ref_mAP_src03": V6_REF_MAP,
    "per_class": [
        {"class": names[c], "n_gt": int(n_gt[c]),
         **{tag: _f(pac[c]) for tag, pac in variants.items()}}
        for c in range(num_classes)
    ],
}
SUMMARY_JSON = "/kaggle/working/medsam_phase0_summary.json"
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print("Outputs written to /kaggle/working:")
for p in ["medsam_phase0_results.csv", "medsam_phase0_summary.json"]:
    fp = Path("/kaggle/working") / p
    print(f"  [{'OK' if fp.exists() else 'MISSING'}] {p}")
print("\naggregate (all classes):")
print(json.dumps(summary["aggregate_mAP50_95"], indent=2))
print("\nlarge classes (Abrasion/Crown/Filling):")
print(json.dumps(summary["large_class_mAP50_95"], indent=2))